# Knowledge Graph RAG

<img src="./media/graph_start.png" width=600>

*[Improving Knowledge Graph Completion with Generative LM and neighbors](https://deeppavlov.ai/research/tpost/bn15u1y4v1-improving-knowledge-graph-completion-wit)*

In the evolving landscape of AI and information retrieval, knowledge graphs have emerged as a powerful way to represent complex, interconnected information. A knowledge graph is a knowledge base that uses a graph-structured data model or topology to represent and operate on data. Knowledge graphs are often used to store interlinked descriptions of entities – objects, events, situations or abstract concepts – while also encoding the free-form semantics or relationships underlying these entities. [Source: Wikipedia](https://en.wikipedia.org/wiki/Knowledge_graph)

What makes knowledge graphs particularly powerful is their ability to mirror human cognition in data. They more explicitly map the relationships between objects, concepts, or ideas together through both their semantic and relational connections. This approach closely parallels how our brains naturally understand and internalize information – not as isolated facts, but as a web of interconnected concepts and relationships.

<img src="./media/coffee_graph_ex.png" width=400>

Looking at a concept like "coffee," we don't just know it's a beverage; we automatically connect it to related concepts like beans, brewing methods, caffeine, morning routines, and social interactions. Knowledge graphs capture these natural associations in a structured way.

Traditional RAG systems, while effective at semantic similarity-based retrieval, often struggle to capture broader conceptual relationships across text chunks. Knowledge Graph RAG addresses this limitation by introducing a structured, hierarchical approach to information organization and retrieval. By representing data in a graph format, these systems can traverse relationships between concepts, enabling more sophisticated query understanding and response generation. This approach allows for targeted querying along specific relationship paths, handles complex multi-hop questions, and provides clearer reasoning through explicit connection paths. The result is a more nuanced and interpretable system that combines the structured reasoning of knowledge graphs with the natural language capabilities of large language models.

While [knowledge graphs are not a new concept](https://blog.google/products/search/introducing-knowledge-graph-things-not/), their creation has traditionally been a resource-intensive process. Early knowledge graphs were built either through manual curation by domain experts or by converting existing structured data from relational databases. This limited both their scale and adaptability to new domains.

<img src="./media/table_comp.png" width=600>

*[What is a Knowledge Graph (KG)?](https://zilliz.com/learn/what-is-knowledge-graph)*

The introduction of LLMs has transformed this landscape. LLMs' capabilities in NLP, reasoning, and relationship extraction now enable automated construction of knowledge graphs from unstructured text. These models can identify entities, infer relationships, and structure information in ways that previously required extensive manual labor. As a plus, this allows knowledge graphs to be dynamically updated and expanded as new information becomes available, making them more practical and scalable for real-world applications.

To see this in action ourselves, and compare it to traditional vector similarity techniques, we'll take a look at Microsoft's Open Source [GraphRAG](https://microsoft.github.io/graphrag/) and how it works behind the scenes.

---
## 3 Main Components of Knowledge Graphs

**Entity**

<img src="./media/entities.png" width=500>

An Entity is a distinct object, person, place, event, or concept that has been extracted from a chunk of text through LLM analysis. Entities form the nodes of the knowledge graph. During the creation of the knowledge graph, when duplicate entities are found they are merged while preserving their various descriptions, creating a comprehensive representation of each unique entity.

**Relationship**

<img src="./media/relationship.png" width=400>

A Relationship defines a connection between two entities in the knowledge graph. These connections are extracted directly from text units through LLM analysis, alongside entities. Each relationship includes a source entity, target entity, and descriptive information about their connection. When duplicate relationships are found between the same entities, they are merged by combining their descriptions to create a more complete understanding of the connection.

**Community**

<img src="./media/communities.png" width=400>

A Community is a cluster of related entities and relationships identified through hierarchical community detection, generally using the [Leiden Algorithm](https://en.wikipedia.org/wiki/Leiden_algorithm). Communities create a structured way to understand different levels of granularity within the knowledge graph, from broad overviews at the top level to detailed local clusters at lower levels. This hierarchical structure helps in organizing and navigating complex knowledge graphs.

---
## GraphRAG Creation Data Flow

<img src=./media/graph_building.png width=1000>

Indexxing in GraphRAG is an extensive process, where we load the document, split it into chunks, create sub graphs at a chunk level, combine these subgraphs into our final graph, algorithmically identify communities, then document the communities main features.

### **Loading and Splitting Our Text**

For our example, we'll be using [The Ultimate Guide to Fine-Tuning LLMs from Basics to Breakthroughs: An Exhaustive Review of Technologies, Research, Best Practices, Applied Research Challenges and Opportunities](https://arxiv.org/pdf/2408.13296).

This will be loaded as a text file (remove index, glossary, and references) and split into 1200 token, 100 token overlap chunks.

In [ ]:
import json
with open("../anchored_index_output.json", "r") as f:
    data = json.load(f)
headers=data["sheets"]['Salary List']["headers"]
anchors = data["sheets"]["Salary List"]["anchors"]
chunks=[]
for chunk_name, chunk_info in anchors.items():
    chunk_info['headers']=headers
    chunks.append(chunk_info)
print(f"Loaded {len(chunks)} chunks")
texts=chunks
chunks


Loaded 19 chunks


[{'range': 'A1:A20',
  'start_row': 1,
  'end_row': 20,
  'summary': 'This data chunk contains information about various companies, their roles, hourly rates, benefits, and years of employment.',
  'context': 'The data includes details on compensation and benefits for co-op positions and internships across different companies.',
  'table': ['Company / Role [A1]\tBye World [B1]\tBenefits [C1]\tYear [D1]\tPosition [E1]',
   '1password [A2]\t50/hr (4th coop) [B2]\t2023 [D2]',
   'Abbott ADD [A3]\t22.05/hr (2nd coop) [B3]',
   'Abnormal Security [A4]\t106k USD annual prorated (~$51/hr USD) [B4]\t5k relocation [C4]',
   'Accedo [A5]\t25-27/hr (3rd coop) [B5]',
   '20-25/hr [B6]',
   'Addepar [A7]\t40/hr, 48/hr (5th coop) [B7]',
   'AdHawk Microsystems [A8]\t40/hr (5th coop) [B8]\t2024 [D8]\tAlgorithms Developer [E8]',
   'less than 44/hr [B9]',
   'AGF Investments [A10]\t25/hr (2nd coop) [B10]\t2023 [D10]\tWeb Developer (Junior) [E10]',
   'AI Arena [A11]\t45/hr [B11]',
   'Akuna Capital [A

**Entity and Relationship Extraction Prompt**

This is a [tuned](https://microsoft.github.io/graphrag/prompt_tuning/auto_prompt_tuning/) entity extraction prompt used in our real GraphRAG implementation, extracted in this format to see what's happening.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.0, model="gpt-4o")
import subprocess, sys

cmd = [
    sys.executable, "-m", "graphrag", "prompt-tune",
    "--config", "settings.yaml",
    "--output", "llm_tuned",
    "--limit", "10",
    "--discover-entity-types",
]

prompt_template=open('llm_tuned/extract_graph.txt').read()
prompt_template3=prompt_template.replace('{','{{').replace('}','}}').replace('{{input_text}}','{input_text}')
print(prompt_template3)
prompt = ChatPromptTemplate.from_template(prompt_template3)
chain = prompt | llm | StrOutputParser()


-Goal-
Given a text document that is potentially relevant to this activity and a list of entity types, identify all entities of those types from the text and all relationships among the identified entities.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types: [ID, value, flag, metric, binary indicator]
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"{{tuple_delimiter}}<entity_name>{{tuple_delimiter}}<entity_type>{{tuple_delimiter}}<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified

**Creating a Response**

In [ ]:
response = chain.invoke({"input_text": texts[10]})
print(response)


2025-10-05 01:43:00,190 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
import re, os
from langchain_neo4j import Neo4jGraph

# --- Neo4j connection ---
os.environ["NEO4J_URI"] = "neo4j://127.0.0.1:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "password"

graph = Neo4jGraph(refresh_schema=False)
graph.query("MATCH (n) DETACH DELETE n")

# --- Fixed parser ---
def parse_data(data):
    data = (
        data.replace("{tuple_delimiter}", "|")
            .replace("{record_delimiter}", "\n")
            .replace("{completion_delimiter}", "")
    )
    entities, relationships = [], []

    pattern = re.compile(r'\("([^"]+)"\|([^|]+)\|([^|]+)\|([^|)]+)(?:\|([^|)]+))?\)')
    # Groups:
    # 1 = kind ("entity" or "relationship")
    # 2.. = remaining tokens (3 or 4)

    for line in data.splitlines():
        line = line.strip()
        if not line:
            continue

        m = pattern.match(line)
        if not m:
            continue

        kind = m.group(1)
        t1, t2, t3, t4, t5 = m.groups()

        if kind == "entity":
            entities.append({
                "name": t2,
                "type": t3,
                "description": t4
            })
        elif kind == "relationship":
            relationships.append({
                "source": t2,
                "target": t3,
                "description": t4,
                "weight": t5
            })

    return entities, relationships


# --- Example input ---
data = """("entity"{tuple_delimiter}CLG0065{tuple_delimiter}ID{tuple_delimiter}Identifier for a data record)
{record_delimiter}
("entity"{tuple_delimiter}76{tuple_delimiter}value{tuple_delimiter}Value associated with CLG0065 in some metric)
{record_delimiter}
("relationship"{tuple_delimiter}CLG0095{tuple_delimiter}5.87{tuple_delimiter}5.87 is a metric value associated with the ID CLG0095{tuple_delimiter}10)
{completion_delimiter}"""
def insert_data(data):
    entities, relationships = parse_data(data)

    print("Entities:", entities)
    print("Relationships:", relationships)

    # --- Insert into Neo4j ---
    for e in entities:
        graph.query("""
            MERGE (n:Entity {name: $name})
            SET n.type = $type, n.description = $description
        """, params=e)

    for r in relationships:
        if r.get("weight") in [None, ""]:
            r['weight']= "unknown"

        graph.query("""
            MERGE (a:Entity {name: $source})
            MERGE (b:Entity {name: $target})
            MERGE (a)-[rel:RELATED_TO {description: $description, weight: $weight}]->(b)
        """, params=r)

def insert_whole_doc(chunks):
    for i,chunk in enumerate(chunks):
        response = chain.invoke({"input_text": chunk})
        insert_data(response)
        print(f"✅ Graph{i/len(chunks)} successfully inserted into Neo4j!")

insert_whole_doc(texts)
# insert_data(response)

2025-10-05 02:09:24,082 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Entities: [{'name': '1password', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly rates'}, {'name': '50/hr (4th coop)', 'type': 'value', 'description': 'Hourly rate for a 4th co-op position at 1password'}, {'name': '2023', 'type': 'value', 'description': 'Year associated with the co-op position at 1password'}, {'name': 'Abbott ADD', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly rates'}, {'name': '22.05/hr (2nd coop)', 'type': 'value', 'description': 'Hourly rate for a 2nd co-op position at Abbott ADD'}, {'name': 'Abnormal Security', 'type': 'ID', 'description': 'A company offering co-op positions with specified annual salary and relocation benefits'}, {'name': '106k USD annual prorated (~$51/hr USD)', 'type': 'value', 'description': 'Annual salary prorated to an hourly rate for a position at Abnormal Security'}, {'name': '5k relocation', 'type': 'value', 'description': 'Relocation benefit offered by Abnormal

2025-10-05 02:09:44,453 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Entities: [{'name': 'Arctic Wolf', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly rates for different co-op years'}, {'name': '36/hr', 'type': 'value', 'description': 'Hourly rate for 3rd co-op at Arctic Wolf'}, {'name': '42/hr', 'type': 'value', 'description': 'Hourly rate for 4th co-op at Arctic Wolf'}, {'name': '2023', 'type': 'value', 'description': 'Year associated with the compensation rate at Arctic Wolf'}, {'name': 'Aterlo Networks', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly rates'}, {'name': '30/hr', 'type': 'value', 'description': 'Hourly rate for 4th co-op at Aterlo Networks'}, {'name': 'Athelas', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly rates and additional benefits'}, {'name': '33/hr USD', 'type': 'value', 'description': 'Hourly rate for co-op at Athelas in USD'}, {'name': '1200/mo USD housing stipend', 'type': 'value', 'description': 'Monthly 

2025-10-05 02:10:05,646 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Entities: [{'name': 'TD', 'type': 'ID', 'description': 'Identifier for a company offering co-op positions'}, {'name': '31/hr (3rd coop)', 'type': 'value', 'description': 'Hourly rate for a third co-op position at TD'}, {'name': '2024', 'type': 'value', 'description': 'Year associated with the co-op position at TD'}, {'name': 'SWE', 'type': 'flag', 'description': 'Flag indicating the role is Software Engineering'}, {'name': 'Teledyne Dalsa', 'type': 'ID', 'description': 'Identifier for a company offering co-op positions'}, {'name': 'Coop Average', 'type': 'value', 'description': 'Average co-op salary at Teledyne Dalsa'}, {'name': 'Tesla', 'type': 'ID', 'description': 'Identifier for a company offering co-op positions'}, {'name': '32/hr (remote), 28-32 USD/hr (onsite, Fremont/Palo Alto)', 'type': 'value', 'description': 'Hourly rate for remote and onsite co-op positions at Tesla'}, {'name': '~4k relocation (onsite, Fremont/Palo Alto)', 'type': 'value', 'description': 'Relocation package 

2025-10-05 02:15:42,863 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Entities: [{'name': 'TTC (Toronto Transit Commission)', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly pay rates'}, {'name': '27.65/hr', 'type': 'value', 'description': 'Hourly pay rate for the 6th co-op term at TTC'}, {'name': 'Trend Micro', 'type': 'ID', 'description': 'A company offering co-op positions with specified hourly pay rates and additional details'}, {'name': '32-34/hr', 'type': 'value', 'description': 'Hourly pay rate range for the 3rd co-op term at Trend Micro'}, {'name': '2024', 'type': 'value', 'description': 'Year associated with the co-op position at Trend Micro'}, {'name': 'SWE', 'type': 'flag', 'description': 'Software Engineering position associated with the co-op at Trend Micro'}, {'name': 'Trexo Robotics', 'type': 'ID', 'description': 'A company offering co-op positions with pay rates based on a percentage above the average'}, {'name': 'co-op average + 20%', 'type': 'value', 'description': 'Pay rate for co-op positions at 

2025-10-05 02:16:00,784 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Entities: [{'name': 'Versett', 'type': 'ID', 'description': 'A company offering co-op positions with specific hourly compensation rates'}, {'name': '32-25/hr', 'type': 'value', 'description': 'Hourly compensation rate for Versett during the 2nd co-op'}, {'name': 'VerticalScope', 'type': 'ID', 'description': 'A company offering co-op positions with specific hourly compensation rates'}, {'name': '30/hr', 'type': 'value', 'description': 'Hourly compensation rate for VerticalScope during the 4th co-op'}, {'name': 'Voiceflow', 'type': 'ID', 'description': 'A company offering co-op positions with variable hourly compensation rates based on term and team'}, {'name': '40-55/hr', 'type': 'value', 'description': 'Hourly compensation rate range for Voiceflow based on term and team'}, {'name': 'Voiceform', 'type': 'ID', 'description': 'A company offering co-op positions with variable hourly compensation rates depending on experience'}, {'name': '20-50/hr', 'type': 'value', 'description': 'Hourly c

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

# connect to Neo4j
graph = Neo4jGraph(
    url="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="password"
)

llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

chain = GraphCypherQAChain.from_llm(
    llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

rephrase_prompt = PromptTemplate.from_template(
    "Convert the question into a concise graph search phrase:\nQuestion: {question}\nSearch phrase:"
)

rephraser = LLMChain(llm=llm, prompt=rephrase_prompt)

# --- Function with feedback loop ---
def query_graph_with_rephrase(user_input, max_attempts=3):
    attempt = 0
    while attempt < max_attempts:
        graph_query = rephraser.run({"question": user_input}).strip()
        print(f"\n Attempt {attempt+1}: Graph query → {graph_query}")
        
        response = chain.invoke({"query": graph_query})
        result = response.get("result", "").strip()

        if result and "don't know" not in result.lower():
            print("\n Found answer:")
            return result
        else:
            user_input = f"Try a different phrasing for: {graph_query}"
            attempt += 1
    
    return " No valid result after rephrasing attempts."

# --- Run ---
user_input = "what is Microsoft pay"
final_result = query_graph_with_rephrase(user_input)
print("\nFinal Result:", final_result)

2025-10-06 00:46:39,169 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



 Attempt 1: Graph query → Microsoft pay overview


> Entering new GraphCypherQAChain chain...


2025-10-06 00:46:40,910 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated Cypher:
cypher
MATCH (e:Entity {name: "Microsoft"})-[:RELATED_TO]->(p:Entity {type: "pay overview"})
RETURN p.description

Full Context:
[]


2025-10-06 00:46:41,728 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...


 Attempt 3: Graph query → "Microsoft salary overview"


> Entering new GraphCypherQAChain chain...


2025-10-06 00:46:47,922 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated Cypher:
MATCH (e1:Entity {name: "Microsoft"})-[:RELATED_TO]->(e2:Entity {type: "salary overview"}) RETURN e2.name, e2.description
Full Context:
[]


2025-10-06 00:46:48,692 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



> Finished chain.

Final Result:  No valid result after rephrasing attempts.


We see the extraction of **entities**:

`("entity"{tuple_delimiter}EVALUATION METRICS{tuple_delimiter}evaluation metrics{tuple_delimiter}Evaluation metrics are criteria  used to assess the performance of AI models, including metrics like cross-entropy, perplexity, factuality, and context relevance)`

As well as **relationships**:

`("relationship"{tuple_delimiter}EVALUATION METRICS{tuple_delimiter}CONTEXT RELEVANCE{tuple_delimiter}Context relevance is an evaluation metric that ensures the model uses the most pertinent information for generating responses{tuple_delimiter}8)`

Following this, these per chunk subgraphs are merged together - any entities with the same name and type are merged by creating an array of their descriptions. Similarly, any relationships with the same source and target are merged by creating an array of their descriptions. These lists are then summarized one more time 

### **Looking at Final Entities and Relationships**

In [77]:
import pandas as pd

entities = pd.read_parquet('./ragtest/output/create_final_entities.parquet')

entities.head()

,id,human_readable_id,title,type,description,text_unit_ids
0,e3a7f24b-88b6-4481-b3a7-c35075a9671f,0,GPT-3,ORGANIZATION,GPT-3 is a large language model developed by O...,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
1,f55cae4e-dd0d-47a2-912b-f7680147dd31,1,GPT-4,ORGANIZATION,GPT-4 is an advanced large language model deve...,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
2,f3e3e46b-6746-45a7-9a26-1432f14c45e4,2,BERT,ORGANIZATION,"BERT, which stands for Bidirectional Encoder R...",[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
3,0491a417-2e18-41c4-ae1e-3e39bf2eb98f,3,PALM,ORGANIZATION,PaLM is a large language model developed by Go...,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
4,2b7f14f5-d1d5-49f6-bace-46fd1767f99e,4,LLAMA,ORGANIZATION,LLAMA is a versatile and advanced model known ...,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...


In [78]:
relationships = pd.read_parquet('./ragtest/output/create_final_relationships.parquet')

relationships.head()

,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,b895553a-f860-4d15-bba2-a42f1464e810,0,GPT-3,GPT-4,"GPT-4 is an advanced version of GPT-3, buildin...",8.0,20,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
1,1548feb2-5a6a-43e6-ab44-81252056193e,1,GPT-3,CHATGPT,"ChatGPT is based on the GPT architecture, spec...",7.0,14,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
2,e538721d-c023-4994-b918-1efece80ea7e,2,GPT-3,BERT,Both BERT and GPT-3 are pre-trained language m...,6.0,18,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
3,063a2941-df53-4b5c-a66a-45e60bdba604,3,GPT-3,REINFORCEMENT LEARNING FROM HUMAN FEEDBACK (RLHF),RLHF is used in training GPT-3 to refine its o...,7.0,13,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...
4,7e0cd0b4-4688-434f-a194-f2b9ce397a94,4,GPT-3,PROMPT ENGINEERING,Prompt engineering is a technique used to guid...,6.0,14,[ca73c495111f5cadd87e6a7a01aed66647ae6623fdf41...


### **Community Detection & Node Embedding**

<img src="./media/leidan.png" width=600>

After we have our basic graph with entities and relationships, we analyze its structure in two ways. Community Detection uses the [Leiden algorithm](https://en.wikipedia.org/wiki/Leiden_algorithm) to find explicit groupings in the graph, creating a hierarchy of related entities. The lower in the hierarchy, the more granular the community. Node Embedding uses [Node2Vec](https://arxiv.org/abs/1607.00653) to create vector representations of each entity, capturing implicit relationships in the graph structure. These complementary approaches let us understand both obvious connections through communities and subtle patterns through embeddings.

Combining all of this with our relationships gives us our final nodes.

In [80]:
nodes = pd.read_parquet('./ragtest/output/create_final_nodes.parquet')

nodes.head(10)

,id,human_readable_id,title,community,level,degree,x,y
0,e3a7f24b-88b6-4481-b3a7-c35075a9671f,0,GPT-3,8,0,12,-4.875545,4.017587
1,e3a7f24b-88b6-4481-b3a7-c35075a9671f,0,GPT-3,43,1,12,-4.875545,4.017587
2,f55cae4e-dd0d-47a2-912b-f7680147dd31,1,GPT-4,8,0,8,-4.561064,1.505724
3,f55cae4e-dd0d-47a2-912b-f7680147dd31,1,GPT-4,46,1,8,-4.561064,1.505724
4,f3e3e46b-6746-45a7-9a26-1432f14c45e4,2,BERT,8,0,6,-5.710580,3.546957
5,f3e3e46b-6746-45a7-9a26-1432f14c45e4,2,BERT,44,1,6,-5.710580,3.546957
6,0491a417-2e18-41c4-ae1e-3e39bf2eb98f,3,PALM,8,0,3,-5.309392,1.548029
7,0491a417-2e18-41c4-ae1e-3e39bf2eb98f,3,PALM,46,1,3,-5.309392,1.548029
8,2b7f14f5-d1d5-49f6-bace-46fd1767f99e,4,LLAMA,3,0,4,-6.644573,0.421999
9,2b7f14f5-d1d5-49f6-bace-46fd1767f99e,4,LLAMA,27,1,4,-6.644573,0.421999


At this step the graph is effectively created, however we can introduce a few extra steps that will allow us to do some advanced retrieval.

### Community Report Generation & Summarization

Now that we have clear community grouping, we can aggregate the main concepts across hierarchical node communities with another generation step, and a shorthand summary of that summary. Similar to the nodes, these summaries are also ran through an embedding model and stored in a vector store.

In [81]:
community_reports = pd.read_parquet('./ragtest/output/create_final_community_reports.parquet')

community_reports.head()

,id,human_readable_id,community,parent,level,title,summary,full_content,rank,rank_explanation,findings,full_content_json,period,size
0,a85d59a64a054114982b1ce6e1ced591,61,61,32,2,Amazon Bedrock and AI Model Providers,The community is centered around Amazon Bedroc...,# Amazon Bedrock and AI Model Providers\n\nThe...,8.5,The impact severity rating is high due to Amaz...,[{'explanation': 'Amazon Bedrock is a pivotal ...,"{\n ""title"": ""Amazon Bedrock and AI Model P...",2024-12-18,9
1,6aafc6eeddd848bc8ffbfb9177790c26,62,62,32,2,AWS and SageMaker JumpStart,The community is centered around Amazon Web Se...,# AWS and SageMaker JumpStart\n\nThe community...,8.5,The impact severity rating is high due to AWS'...,[{'explanation': 'Amazon Web Services (AWS) is...,"{\n ""title"": ""AWS and SageMaker JumpStart"",...",2024-12-18,2
2,e13e3ed0a0b74fd090319957ae9f3e1e,14,14,0,1,PPO for LLM Alignment and Reinforcement Learni...,The community centers around the study 'PPO fo...,# PPO for LLM Alignment and Reinforcement Lear...,7.5,The impact severity rating is high due to the ...,[{'explanation': 'The study 'PPO for LLM Align...,"{\n ""title"": ""PPO for LLM Alignment and Rei...",2024-12-18,7
3,828baab1461b439ea71203ad8fd0aae5,15,15,0,1,HuggingFace and Advanced NLP Tools,"The community is centered around HuggingFace, ...",# HuggingFace and Advanced NLP Tools\n\nThe co...,8.5,The impact severity rating is high due to Hugg...,[{'explanation': 'HuggingFace is a prominent e...,"{\n ""title"": ""HuggingFace and Advanced NLP ...",2024-12-18,7
4,791da6e7031e45228442b277e7d912c6,16,16,0,1,OpenAI and AI Development Platforms,"The community is centered around OpenAI, a lea...",# OpenAI and AI Development Platforms\n\nThe c...,8.5,The impact severity rating is high due to the ...,[{'explanation': 'OpenAI is a central entity i...,"{\n ""title"": ""OpenAI and AI Development Pla...",2024-12-18,7


In [83]:
print(community_reports["full_content"][0])

# Amazon Bedrock and AI Model Providers

The community is centered around Amazon Bedrock, a service by AWS that facilitates access to foundation models from various AI innovators. Key entities include AI21 Labs, Anthropic, Cohere, Mistral AI, and Stability AI, all of which provide models accessible through Amazon Bedrock. The service integrates with AWS infrastructure, including AWS Lambda and AWS SageMaker, to support scalable AI model deployment.

## Amazon Bedrock as a central service

Amazon Bedrock is a pivotal service within the AWS ecosystem, designed to simplify access to high-performing foundation models for generative AI applications. It integrates seamlessly with other AWS services, such as Amazon S3, AWS Lambda, and AWS SageMaker, to facilitate the fine-tuning and deployment of AI models. This integration underscores its importance in the AI landscape, providing a comprehensive suite of tools for scalable AI model deployment [Data: Entities (206); Relationships (281, 326, 3

In [82]:
print(community_reports["summary"][0])

The community is centered around Amazon Bedrock, a service by AWS that facilitates access to foundation models from various AI innovators. Key entities include AI21 Labs, Anthropic, Cohere, Mistral AI, and Stability AI, all of which provide models accessible through Amazon Bedrock. The service integrates with AWS infrastructure, including AWS Lambda and AWS SageMaker, to support scalable AI model deployment.


### The Final Graph!

<img src="./media/ghraphrag_viz.svg" width=800>

*[Full Size PDF](./ghraphrag_viz.pdf)*

---

## GraphRAG Retrieval

<img src="./media/kg_retrieval.png" width=600>

*[Unifying Large Language Models and Knowledge Graphs: A Roadmap](https://arxiv.org/pdf/2306.08302)*

With our knowledge graph constructed, and hierarchichal communities delineated, we can now perform multiple types of search that can both take advantage of the graph structure, and multiple levels of specificity across our communities. Specifically:

1. **Global Search**: Uses the LLM Generated community reports from a specified level of the graph's community hierarchy as context data to generate response.
2. **Local Search**: Combines structured data from the knowledge graph with unstructured data from the input document(s) to augment the LLM context with relevant entity information.
3. **Drift Search**: Dynamic Reasoning and Inference with Flexible Traversal, an approach to local search queries by including community information in the search process, thus combining global and local search.

**GraphRAG Retrieval Function**

*Note: Wrapping the [GraphRAG CLI tool](https://microsoft.github.io/graphrag/cli/) as a function here instead of using their [library](https://microsoft.github.io/graphrag/examples_notebooks/api_overview/) for an easier example. As such, notebook needs to be running in the same GraphRAG environment/kernal.*

In [84]:
import subprocess
import shlex
from typing import Optional

def query_graphrag(
    query: str,
    method: str = "global",
    root_path: str = "./ragtest",
    timeout: Optional[int] = None,
    community_level: int = 2,
    dynamic_community_selection: bool = False
) -> str:
    """
    Execute a GraphRAG query using the CLI tool.
    
    Args:
        query (str): The query string to process
        method (str): Query method (e.g., "global", "local", or "drift")
        root_path (str): Path to the root directory
        timeout (int, optional): Timeout in seconds for the command
        community_level (int): The community level in the Leiden community hierarchy (default: 2)
        dynamic_community_selection (bool): Whether to use global search with dynamic community selection (default: False)
    
    Returns:
        str: The output from GraphRAG
        
    Raises:
        subprocess.CalledProcessError: If the command fails
        subprocess.TimeoutExpired: If the command times out
        ValueError: If community_level is negative
    """
    # Validate community level
    if community_level < 0:
        raise ValueError("Community level must be non-negative")
    
    # Construct the base command
    command = [
        'graphrag', 'query',
        '--root', root_path,
        '--method', method,
        '--query', query,
        '--community-level', str(community_level)
    ]
    
    # Add dynamic community selection flag if enabled
    if dynamic_community_selection:
        command.append('--dynamic-community-selection')
    
    try:
        # Execute the command and capture output
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        
        # Check if the command was successful
        result.check_returncode()
        
        return result.stdout.strip()
        
    except subprocess.CalledProcessError as e:
        error_message = f"Command failed with exit code {e.returncode}\nError: {e.stderr}"
        raise subprocess.CalledProcessError(
            e.returncode,
            e.cmd,
            output=e.output,
            stderr=error_message
        )

### Local Search

<img src="./media/local_search.png" width=900>

The GraphRAG approach to local search is the most similar to regular semantic RAG search. It combines structured data from the knowledge graph with unstructured data from the input documents to augment the LLM context with relevant entity information. In essence, we are going to first search for relevant entities to the query using semantic search. These become the entry points on our graph that we can now traverse. Starting at these points, we look at connected chunks of text, community reports, other entities, and relationships between them. All of the data retrieved is filtered and ranked to fit into a pre-defined context window.

In [88]:
result = query_graphrag(
    query="How does a company choose between RAG, fine-tuning, and different PEFT approaches?",
    method="local"
)
print("Query result:")
print(result)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Query result:
INFO: Vector Store Args: {
    "type": "lancedb",
    "db_uri": "/Users/adamlucek/Desktop/github/GraphRAG/ragtest/output/lancedb",
    "container_name": "==== REDACTED ====",
    "overwrite": true
}
creating llm client with {'api_key': 'REDACTED,len=51', 'type': "openai_chat", 'encoding_model': 'cl100k_base', 'model': 'gpt-4o', 'max_tokens': 4000, 'temperature': 0.0, 'top_p': 1.0, 'n': 1, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'request_timeout': 180.0, 'api_base': None, 'api_version': None, 'organization': None, 'proxy': None, 'audience': None, 'deployment_name': None, 'model_supports_json': True, 'tokens_per_minute': 0, 'requests_per_minute': 0, 'max_retries': 10, 'max_retry_wait': 10.0, 'sleep_on_rate_limit_recommendation': True, 'concurrent_requests': 25, 'responses': None}
creating embedding llm client with {'api_key': 'REDACTED,len=51', 'type': "openai_embedding", 'encoding_model': 'cl100k_base', 'model': 'text-embedding-3-small', 'max_tokens': 4000, 'tem

### Global Search

<img src="./media/global_search.png" width=1000>

Through the semantic clustering of communities during the indexxing process outlined above we created community reports as summaries of high level themes across these groupings. Having this community summary data at various levels allows us to do something that traditional RAG performs poorly at, answering queries about broad themes and ideas across our unstructured data.

To capture as much broad information as possible in an efficient manner, GraphRAG implements a [map reduce](https://en.wikipedia.org/wiki/MapReduce) approach. Given a query, relevant community node reports at a specific hierarchical level are retrieved. These are shuffled and chunked, where each chunk is used to generate a list of points that each have their own "importance score". These intermediate points are ranked and filtered, attempting to maintain the most important points. These become the aggregate intermediary response, which is passed to the LLM as the context for the final response.

In [86]:
result = query_graphrag(
    query="How does a company choose between RAG, fine-tuning, and different PEFT approaches?",
    method="global"
)
print("Query result:")
print(result)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Query result:
creating llm client with {'api_key': 'REDACTED,len=51', 'type': "openai_chat", 'encoding_model': 'cl100k_base', 'model': 'gpt-4o', 'max_tokens': 4000, 'temperature': 0.0, 'top_p': 1.0, 'n': 1, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'request_timeout': 180.0, 'api_base': None, 'api_version': None, 'organization': None, 'proxy': None, 'audience': None, 'deployment_name': None, 'model_supports_json': True, 'tokens_per_minute': 0, 'requests_per_minute': 0, 'max_retries': 10, 'max_retry_wait': 10.0, 'sleep_on_rate_limit_recommendation': True, 'concurrent_requests': 25, 'responses': None}

SUCCESS: Global Search Response:
### Choosing Between RAG, Fine-Tuning, and PEFT Approaches

When a company is deciding between Retrieval-Augmented Generation (RAG), fine-tuning, and Parameter-Efficient Fine-Tuning (PEFT) approaches, several key factors must be considered. These factors include the specific requirements of the application, the need for external data integration, co

### DRIFT Search

<img src="./media/drift_search.png" width=1000>

[Dynamic Reasoning and Inference with Flexible Traversal](https://www.microsoft.com/en-us/research/blog/introducing-drift-search-combining-global-and-local-search-methods-to-improve-quality-and-efficiency/), or DRIFT, is a novel GraphRAG concept introduced by Microsoft as an approach to local search queries that include community information in the search process.

The user's query is initially processed through [Hypothetical Document Embedding (HyDE)](https://arxiv.org/pdf/2212.10496), which creates a hypothetical document similar to those found in the graph already, but using the user's topic query. This document is embedded and used for semantic retrieval of the top-k relevant community reports. From these matches, we generate an initial answer along with several follow-up questions as a lightweight version of global search. They refer to this as the primer.

Once this primer phase is complete, we execute local searches for each follow-up question generated. Each local search produces both intermediate answers and new follow-up questions, creating a refinement loop. This loop runs for two iterations (noted future research planned to develop reward functions for smarter termination). An important note that makes these local searches unique is that they are informed by both community-level knowledge and detailed entity/relationship data. This allows the DRIFT process to find relevant information even when the initial query diverges from the indexing persona, and it can adapt its approach based on emerging information during the search.

The final output is structured as a hierarchy of questions and answers, ranked by their relevance to the original query. Map reduce is used again with an equal weighting on all intermediate answers, then passed to the language model for a final response. DRIFT cleverly combines global and local search with guided exploration to provide both broad context and specific details in responses.

In [89]:
result = query_graphrag(
    query="How does a company choose between RAG, fine-tuning, and different PEFT approaches?",
    method="drift"
)
print("Query result:")
print(result)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Query result:
INFO: Vector Store Args: {
    "type": "lancedb",
    "db_uri": "/Users/adamlucek/Desktop/github/GraphRAG/ragtest/output/lancedb",
    "container_name": "==== REDACTED ====",
    "overwrite": true
}
creating llm client with {'api_key': 'REDACTED,len=51', 'type': "openai_chat", 'encoding_model': 'cl100k_base', 'model': 'gpt-4o', 'max_tokens': 4000, 'temperature': 0.0, 'top_p': 1.0, 'n': 1, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'request_timeout': 180.0, 'api_base': None, 'api_version': None, 'organization': None, 'proxy': None, 'audience': None, 'deployment_name': None, 'model_supports_json': True, 'tokens_per_minute': 0, 'requests_per_minute': 0, 'max_retries': 10, 'max_retry_wait': 10.0, 'sleep_on_rate_limit_recommendation': True, 'concurrent_requests': 25, 'responses': None}
creating embedding llm client with {'api_key': 'REDACTED,len=51', 'type': "openai_embedding", 'encoding_model': 'cl100k_base', 'model': 'text-embedding-3-small', 'max_tokens': 4000, 'tem

---

## Comparing to Regular Vector Database Retrieval

<img src="./media/basic_retrieval.png" width=600>
 
To give some comparison, let's look back at traditional chunking, embedding, and similarity retrieval RAG

**Instantiate our Database**

For this we'll be using [ChromaDB](https://www.trychroma.com) with the same chunks as were loaded into our graph.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./notebook/chromadb")
paper_collection = chroma_client.get_or_create_collection(name="paper_collection")

**Embed Chunks Into Collection**

In [ ]:
i = 0
for text in texts:
    paper_collection.add(
        documents=[text],
        ids=f"chunk_{i}"
    )
    i += 1

**Retrieval Function**

In [ ]:
def chroma_retrieval(query, num_results=5):
    results = paper_collection.query(
        query_texts=[query],
        n_results=num_results
    )
    return results

**RAG Prompt & Chain**

In [90]:
rag_prompt_template = """
Generate a response of the target length and format that responds to the user's question, summarizing all information in the input data tables appropriate for the response length and format, and incorporating any relevant general knowledge.

If you don't know the answer, just say so. Do not make anything up.

Do not include information where the supporting evidence for it is not provided.

Context: {retrieved_docs}

User Question: {query}

"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

rag_chain = rag_prompt | llm | StrOutputParser()

**RAG Function**

In [91]:
def chroma_rag(query):
    retrieved_docs = chroma_retrieval(query)["documents"][0]
    response = rag_chain.invoke({"retrieved_docs": retrieved_docs, "query": query})
    return response

**RAG Response**

In [92]:
response = chroma_rag("How does a company choose between RAG, fine-tuning, and different PEFT approaches?")
print(response)

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"

When choosing between Retrieval-Augmented Generation (RAG), fine-tuning, and different Parameter-Efficient Fine-Tuning (PEFT) approaches, a company should consider several factors:

1. **Data Access and Updates**: RAG is preferable for applications requiring access to external data sources or environments where data frequently updates. It provides dynamic data retrieval capabilities and is less prone to generating incorrect information.

2. **Model Behavior and Domain-Specific Knowledge**: Fine-tuning is suitable when the model needs to adjust its behavior, writing style, or incorporate domain-specific knowledge. It is effective if there is ample domain-specific, labeled training data available.

3. **Resource Constraints and Efficiency**: PEFT approaches like LoRA and DEFT are designed to reduce computational and resource requirements. LoRA focuses on low-rank matrices to reduce memory usage and computational load, while DEFT optimizes the fine-tuning process by focusing on the most c

---
## Discussion

**Traditional/Naive RAG:**

Benefits:
- Simpler implementation and deployment
- Works well for straightforward information retrieval tasks
- Good at handling unstructured text data
- Lower computational overhead

Drawbacks:
- Loses structural information when chunking documents
- Can break up related content during text segmentation
- Limited ability to capture relationships between different pieces of information
- May struggle with complex reasoning tasks requiring connecting multiple facts
- Potential for incomplete or fragmented answers due to chunking boundaries

**GraphRAG:**

Benefits:
- Preserves structural relationships and hierarchies in the knowledge
- Better at capturing connections between related information
- Can provide more complete and contextual answers
- Improved retrieval accuracy by leveraging graph structure
- Better supports complex reasoning across multiple facts
- Can maintain document coherence better than chunk-based approaches
- More interpretable due to explicit knowledge representation

Drawbacks:
- More complex to implement and maintain
- Requires additional processing to construct and update knowledge graphs
- Higher computational overhead for graph operations
- May require domain expertise to define graph schema/structure
- More challenging to scale to very large datasets
- Additional storage requirements for graph structure

**Key Differentiators:**
1. Knowledge Representation: Traditional RAG treats everything as flat text chunks, while GraphRAG maintains structured relationships in a graph format

2. Context Preservation: GraphRAG better preserves context and relationships between different pieces of information compared to the chunking approach of traditional RAG

3. Reasoning Capability: GraphRAG enables better multi-hop reasoning and connection of related facts through graph traversal, while traditional RAG is more limited to direct retrieval

4. Answer Quality: GraphRAG tends to produce more complete and coherent answers since it can access related information through graph connections rather than being limited by chunk boundaries

The choice between traditional RAG and GraphRAG often depends on the specific use case, with GraphRAG being particularly valuable when maintaining relationships between information is important or when complex reasoning is required. An important note as well, GraphRAG approaches still rely on regular embedding and retrieval methods themselves. They compliment eahcother!